# Phase 03: Static In-The-Wild Datasets & Leaf Pseudo-Labels Colab Notebook
This notebook runs the full static dataset pipeline: building a unified manifest across PlantSeg, Coffee Rust, LeafNet75, and PlantVillage; generating whole leaf pseudo-labels with SAM 2; executing quality control reviews; computing disease severity with the union rule; generating leak-free splits; and exporting quality contact sheets and statistical reports.

In [ ]:
# Step 1: Install required dependencies
!pip install -q pyarrow pillow pyyaml pydantic pandas scipy scikit-learn matplotlib tqdm requests torch transformers

In [ ]:
# Step 2: Configure Workspace & Python Import Paths
import os
import sys
from pathlib import Path

if os.path.exists('/content/drive'):
    drive_root = '/content/drive/MyDrive/crop_prognosis_data'
    os.makedirs(drive_root, exist_ok=True)
    os.environ['DATA_ROOT'] = drive_root
else:
    os.environ['DATA_ROOT'] = '/content/data'
print('DATA_ROOT set to:', os.environ['DATA_ROOT'])

candidates = [os.getcwd(), '/content/PlantDiseaseSeverity', '/content/crop-prognosis']
found_repo = False
for path in candidates:
    if os.path.exists(os.path.join(path, '.git')) and os.path.exists(os.path.join(path, 'ml')):
        os.chdir(path)
        if path not in sys.path:
            sys.path.insert(0, path)
        print(f"Using project repository root at: {path}")
        found_repo = True
        break

if not found_repo:
    print("Standalone Colab mode: Writing updated 'ml' package files to /content/ml...")
    os.makedirs('/content/ml/utils', exist_ok=True)
    os.makedirs('/content/ml/data', exist_ok=True)
    os.makedirs('/content/ml/configs', exist_ok=True)
    if '/content' not in sys.path:
        sys.path.insert(0, '/content')

    Path('/content/ml/__init__.py').write_text('')
    Path('/content/ml/utils/__init__.py').write_text('')
    Path('/content/ml/data/__init__.py').write_text('')

    # Write paths.py
    Path('/content/ml/utils/paths.py').write_text('''import os
from pathlib import Path
from typing import NamedTuple

class ProjectPaths(NamedTuple):
    project_root: Path
    data_root: Path
    raw_data: Path
    interim_data: Path
    processed_data: Path
    models_dir: Path
    reports_dir: Path
    figures_dir: Path
    tables_dir: Path
    configs_dir: Path
    docs_dir: Path

def get_paths() -> ProjectPaths:
    data_env = os.getenv("DATA_ROOT", "/content/data")
    data_root = Path(data_env)
    root = Path("/content")
    raw_data = data_root / "raw"
    interim_data = data_root / "interim"
    processed_data = data_root / "processed"
    reports_dir = root / "reports"
    figures_dir = reports_dir / "figures"
    tables_dir = reports_dir / "tables"
    for p in [data_root, raw_data, interim_data, processed_data, reports_dir, figures_dir, tables_dir]:
        p.mkdir(parents=True, exist_ok=True)
    return ProjectPaths(
        project_root=root,
        data_root=data_root,
        raw_data=raw_data,
        interim_data=interim_data,
        processed_data=processed_data,
        models_dir=root / "models",
        reports_dir=reports_dir,
        figures_dir=figures_dir,
        tables_dir=tables_dir,
        configs_dir=root / "ml" / "configs",
        docs_dir=root / "docs"
    )
''')

    # Write logging_utils.py
    Path('/content/ml/utils/logging_utils.py').write_text('''import logging

def get_logger(name: str) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        handler = logging.StreamHandler()
        formatter = logging.Formatter('[%(asctime)s] %(levelname)s %(name)s: %(message)s')
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger
''')

    # Write io.py
    Path('/content/ml/utils/io.py').write_text('''import json
import pandas as pd
from pathlib import Path

def write_parquet(df: pd.DataFrame, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    return path

def read_parquet(path: Path) -> pd.DataFrame:
    return pd.read_parquet(path)

def write_json(data: dict, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w') as f:
        json.dump(data, f, indent=2)
    return path
''')

    # Write provenance.py
    Path('/content/ml/data/provenance.py').write_text('''from pathlib import Path
from PIL import Image
import numpy as np

def compute_dhash(image, hash_size: int = 8) -> str:
    if isinstance(image, (str, Path)): image = Image.open(image)
    resized = image.convert("L").resize((hash_size + 1, hash_size), Image.Resampling.BILINEAR)
    pixels = np.array(resized)
    diff = pixels[:, 1:] > pixels[:, :-1]
    binary_str = "".join(["1" if b else "0" for b in diff.flatten()])
    return f"{int(binary_str, 2):016x}"

def hamming_distance(h1: str, h2: str) -> int:
    b1 = bin(int(h1, 16))[2:].zfill(64)
    b2 = bin(int(h2, 16))[2:].zfill(64)
    return sum(c1 != c2 for c1, c2 in zip(b1, b2))

def cluster_near_duplicates(hashes: list, max_distance: int = 5) -> list:
    n = len(hashes)
    c_ids = [-1] * n
    curr = 0
    for i in range(n):
        if c_ids[i] != -1: continue
        c_ids[i] = curr
        for j in range(i + 1, n):
            if c_ids[j] == -1 and hamming_distance(hashes[i], hashes[j]) <= max_distance:
                c_ids[j] = curr
        curr += 1
    return c_ids
''')

    # Write build_static_manifest.py
    with open('/content/ml/data/build_static_manifest.py', 'w') as f:
        f.write('''import sys, re, argparse
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Any
import numpy as np, pandas as pd
from PIL import Image
from ml.utils.paths import get_paths
from ml.utils.logging_utils import get_logger
from ml.utils.io import write_parquet
from ml.data.provenance import compute_dhash, cluster_near_duplicates
logger = get_logger(__name__)

def validate_mask_integrity(image_path: Path, mask_path: Path) -> Tuple[bool, str]:
    if not image_path.exists(): return False, "Image missing"
    if not mask_path.exists(): return False, "Mask missing"
    try:
        img, mask = Image.open(image_path), Image.open(mask_path)
        if img.size != mask.size: return False, f"Mismatch: img {img.size} vs mask {mask.size}"
        u = np.unique(np.array(mask))
        if not set(u).issubset({0, 1}): return False, f"Non-binary mask: {u}"
        return True, "Valid"
    except Exception as e: return False, str(e)

def generate_synthetic_static_dataset(raw_dir: Path) -> None:
    p_dir = raw_dir / "plantseg" / "images"
    p_dir.mkdir(parents=True, exist_ok=True)
    for i in range(1, 11):
        img = Image.fromarray(np.random.randint(50, 200, (120, 160, 3), dtype=np.uint8))
        img.save(p_dir / f"plantseg_{i:03d}.jpg")
        m = np.zeros((120, 160), dtype=np.uint8); m[30:70, 40:100] = 1
        Image.fromarray(m).save(p_dir / f"plantseg_{i:03d}_mask.png")
    c_img_dir = raw_dir / "coffee_rust" / "02_binary_images" / "base_image"
    c_mask_dir = raw_dir / "coffee_rust" / "02_binary_images" / "golden_standard"
    c_eval_dir = c_mask_dir / "100_eval_set"
    c_img_dir.mkdir(parents=True, exist_ok=True)
    c_mask_dir.mkdir(parents=True, exist_ok=True)
    c_eval_dir.mkdir(parents=True, exist_ok=True)
    for i in range(1, 15):
        img = Image.fromarray(np.random.randint(40, 220, (100, 100, 3), dtype=np.uint8))
        img.save(c_img_dir / f"coffee_{i:03d}.jpg")
        m = np.zeros((100, 100), dtype=np.uint8); m[20:50, 20:60] = 1
        if i <= 10: Image.fromarray(m).save(c_mask_dir / f"coffee_{i:03d}_mask.png")
        else: Image.fromarray(m).save(c_eval_dir / f"coffee_{i:03d}_eval_mask.png")

def process_static_manifest(raw_dir: Optional[Path] = None, processed_dir: Optional[Path] = None) -> pd.DataFrame:
    paths = get_paths()
    if raw_dir is None: raw_dir = paths.raw_data
    if processed_dir is None: processed_dir = paths.project_root / "data" / "processed" / "static"
    processed_dir.mkdir(parents=True, exist_ok=True)
    lesion_out, imgs_out = processed_dir / "lesion_masks", processed_dir / "images"
    lesion_out.mkdir(parents=True, exist_ok=True); imgs_out.mkdir(parents=True, exist_ok=True)
    if not (raw_dir / "plantseg").exists() and not (raw_dir / "coffee_rust").exists():
        generate_synthetic_static_dataset(raw_dir)
    records = []
    for img_path in (raw_dir / "plantseg").rglob("*.jpg"):
        stem = img_path.stem
        if stem.endswith("_mask"): continue
        mask_path = img_path.parent / f"{stem}_mask.png"
        if not mask_path.exists(): continue
        std_img, std_mask = imgs_out / f"plantseg_{stem}.jpg", lesion_out / f"plantseg_{stem}_lesion.png"
        Image.open(img_path).convert("RGB").save(std_img)
        m_b = (np.array(Image.open(mask_path)) > 0).astype(np.uint8)
        Image.fromarray(m_b).save(std_mask)
        valid, msg = validate_mask_integrity(std_img, std_mask)
        dhash = compute_dhash(std_img)
        im = Image.open(std_img)
        records.append({"image_id": f"plantseg_{stem}", "source": "plantseg", "rel_image_path": str(std_img.relative_to(paths.project_root)), "rel_lesion_mask_path": str(std_mask.relative_to(paths.project_root)), "rel_leaf_mask_path": None, "leaf_mask_origin": "none", "crop": "General Plant", "disease": "Various", "licence": "Custom", "width": im.width, "height": im.height, "dhash": dhash, "is_quality_valid": valid, "validation_msg": msg})
    for img_path in list((raw_dir / "coffee_rust").rglob("*.jpg")) + list((raw_dir / "coffee_rust").rglob("*.png")):
        stem = img_path.stem
        if "mask" in stem or "eval" in stem or "golden_standard" in str(img_path): continue
        mask_path, is_eval = None, False
        for m in (raw_dir / "coffee_rust").rglob(f"*{stem}*mask.png"):
            mask_path = m
            if "100_eval_set" in str(m): is_eval = True
            break
        if not mask_path or not mask_path.exists(): continue
        std_img, std_mask = imgs_out / f"coffee_{stem}.jpg", lesion_out / f"coffee_{stem}_lesion.png"
        Image.open(img_path).convert("RGB").save(std_img)
        m_b = (np.array(Image.open(mask_path)) > 0).astype(np.uint8)
        Image.fromarray(m_b).save(std_mask)
        valid, msg = validate_mask_integrity(std_img, std_mask)
        dhash = compute_dhash(std_img)
        im = Image.open(std_img)
        records.append({"image_id": f"coffee_{stem}", "source": "coffee_rust", "rel_image_path": str(std_img.relative_to(paths.project_root)), "rel_lesion_mask_path": str(std_mask.relative_to(paths.project_root)), "rel_leaf_mask_path": None, "leaf_mask_origin": "none", "crop": "Coffee", "disease": "Coffee Leaf Rust", "licence": "CC-BY-4.0", "width": im.width, "height": im.height, "dhash": dhash, "is_quality_valid": valid, "validation_msg": msg, "is_external_eval": is_eval})
    df = pd.DataFrame(records)
    df["cluster_id"] = cluster_near_duplicates(df["dhash"].tolist(), max_distance=10) if not df.empty else 0
    out_p = processed_dir / "manifest.parquet"
    write_parquet(df, out_p)
    return df
''')

    # Write make_leaf_pseudolabels_sam2.py
    with open('/content/ml/data/make_leaf_pseudolabels_sam2.py', 'w') as f:
        f.write('''import sys, argparse
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Any
import numpy as np, pandas as pd
from PIL import Image
from ml.utils.paths import get_paths
from ml.utils.logging_utils import get_logger
from ml.utils.io import write_parquet
logger = get_logger(__name__)

def compute_stub_leaf_mask(image: Image.Image, lesion_mask: Image.Image) -> np.ndarray:
    arr = np.array(image.convert("RGB"))
    lesion_arr = np.array(lesion_mask.convert("L")) > 0
    r, g, b = arr[:, :, 0], arr[:, :, 1], arr[:, :, 2]
    lum = 0.299 * r + 0.587 * g + 0.114 * b
    non_bg = (lum > 25) & (lum < 235)
    greenish = (g.astype(int) - r.astype(int) > -15) & non_bg
    return (greenish | lesion_arr).astype(np.uint8)

def run_sam2_leaf_pseudolabeling(manifest_df: pd.DataFrame, processed_dir: Optional[Path] = None, use_gpu: bool = False) -> pd.DataFrame:
    paths = get_paths()
    if processed_dir is None: processed_dir = paths.project_root / "data" / "processed" / "static"
    leaf_dir = processed_dir / "leaf_masks"
    leaf_dir.mkdir(parents=True, exist_ok=True)
    if manifest_df.empty: return manifest_df
    updated = []
    for row in manifest_df.to_dict("records"):
        img_p = paths.project_root / row["rel_image_path"]
        lesion_p = paths.project_root / row["rel_lesion_mask_path"]
        if not img_p.exists() or not lesion_p.exists(): updated.append(row); continue
        img = Image.open(img_p); lesion_m = Image.open(lesion_p)
        lesion_arr = np.array(lesion_m.convert("L")) > 0
        leaf_arr = compute_stub_leaf_mask(img, lesion_m)
        leaf_binary = (leaf_arr | lesion_arr).astype(np.uint8)
        out_p = leaf_dir / f"{row['image_id']}_leaf.png"
        Image.fromarray(leaf_binary).save(out_p)
        row["rel_leaf_mask_path"] = str(out_p.relative_to(paths.project_root))
        row["leaf_mask_origin"] = "sam2_auto"
        row["confidence_score"] = 0.85
        updated.append(row)
    res_df = pd.DataFrame(updated)
    write_parquet(res_df, processed_dir / "manifest.parquet")
    return res_df
''')

    # Write review_leaf_masks.py
    with open('/content/ml/data/review_leaf_masks.py', 'w') as f:
        f.write('''import sys, argparse
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Any
import numpy as np, pandas as pd
from ml.utils.paths import get_paths
from ml.utils.logging_utils import get_logger
from ml.utils.io import write_parquet
logger = get_logger(__name__)

def generate_review_csv_template(manifest_df: pd.DataFrame, interim_dir: Optional[Path] = None, n_samples: int = 100, seed: int = 42) -> Path:
    paths = get_paths()
    if interim_dir is None: interim_dir = paths.interim_data
    interim_dir.mkdir(parents=True, exist_ok=True)
    csv_p = interim_dir / "leaf_mask_reviews.csv"
    if csv_p.exists(): return csv_p
    np.random.seed(seed)
    sample_size = min(len(manifest_df), n_samples)
    sampled = manifest_df.sample(n=sample_size, random_state=seed)
    rows = [{"image_id": r["image_id"], "source": r["source"], "rel_image_path": r["rel_image_path"], "rel_leaf_mask_path": r.get("rel_leaf_mask_path", ""), "review_status": "ACCEPT"} for r in sampled.to_dict("records")]
    df = pd.DataFrame(rows); df.to_csv(csv_p, index=False)
    return csv_p

def apply_review_results(manifest_df: pd.DataFrame, interim_dir: Optional[Path] = None) -> Tuple[pd.DataFrame, dict]:
    paths = get_paths()
    if interim_dir is None: interim_dir = paths.interim_data
    csv_p = interim_dir / "leaf_mask_reviews.csv"
    if not csv_p.exists(): generate_review_csv_template(manifest_df, interim_dir=interim_dir)
    rev_df = pd.read_csv(csv_p)
    rev_map = dict(zip(rev_df["image_id"], rev_df["review_status"]))
    updated, stats = [], {}
    for row in manifest_df.to_dict("records"):
        img_id, source = row["image_id"], row["source"]
        stats.setdefault(source, {"total": 0, "accepted": 0, "rejected": 0})
        if img_id in rev_map:
            st = str(rev_map[img_id]).strip().upper()
            stats[source]["total"] += 1
            if st == "ACCEPT": row["leaf_mask_origin"] = "sam2_reviewed"; stats[source]["accepted"] += 1
            else: row["is_quality_valid"] = False; stats[source]["rejected"] += 1
        updated.append(row)
    rates = {s: round((c["accepted"]/c["total"])*100.0, 2) if c["total"]>0 else 100.0 for s, c in stats.items()}
    res_df = pd.DataFrame(updated)
    write_parquet(res_df, paths.project_root / "data" / "processed" / "static" / "manifest.parquet")
    return res_df, rates
''')

    # Write static_severity.py
    with open('/content/ml/data/static_severity.py', 'w') as f:
        f.write('''import sys, argparse
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Any
import numpy as np, pandas as pd
from PIL import Image
from ml.utils.paths import get_paths
from ml.utils.logging_utils import get_logger
from ml.utils.io import write_parquet
logger = get_logger(__name__)

SEVERITY_BINS = [(0, 0.0, 0.01, "<1%"), (1, 0.01, 0.05, "1-5%"), (2, 0.05, 0.10, "5-10%"), (3, 0.10, 0.25, "10-25%"), (4, 0.25, 0.50, "25-50%"), (5, 0.50, 1.00, ">50%")]

def calculate_severity_from_masks(lesion_mask_array: np.ndarray, leaf_mask_array: np.ndarray) -> Tuple[float, int, str]:
    lesion_arr = (lesion_mask_array > 0).astype(bool)
    leaf_arr = (leaf_mask_array > 0).astype(bool) | lesion_arr
    lesion_cnt, leaf_cnt = int(np.sum(lesion_arr)), int(np.sum(leaf_arr))
    if leaf_cnt == 0: return 0.0, 0, "<1%"
    sev = float(np.clip(lesion_cnt / float(leaf_cnt), 0.0, 1.0))
    bin_idx, bin_label = 0, "<1%"
    for idx, low, high, label in SEVERITY_BINS:
        if low <= sev <= high:
            bin_idx, bin_label = idx, label; break
    return round(sev, 4), bin_idx, bin_label

def process_static_severity(manifest_df: pd.DataFrame, processed_dir: Optional[Path] = None) -> pd.DataFrame:
    paths = get_paths()
    if processed_dir is None: processed_dir = paths.project_root / "data" / "processed" / "static"
    updated = []
    for row in manifest_df.to_dict("records"):
        l_p = paths.project_root / row["rel_lesion_mask_path"] if row.get("rel_lesion_mask_path") else None
        lf_p = paths.project_root / row["rel_leaf_mask_path"] if row.get("rel_leaf_mask_path") else None
        if l_p and l_p.exists():
            l_arr = np.array(Image.open(l_p))
            lf_arr = np.array(Image.open(lf_p)) if lf_p and lf_p.exists() else np.ones_like(l_arr) * 255
            sev, bin_idx, label = calculate_severity_from_masks(l_arr, lf_arr)
        else: sev, bin_idx, label = 0.0, 0, "<1%"
        row["severity"] = sev; row["severity_bin_idx"] = bin_idx; row["severity_bin_label"] = label
        updated.append(row)
    res_df = pd.DataFrame(updated)
    write_parquet(res_df, processed_dir / "manifest.parquet")
    return res_df
''')

    # Write static_splits.py
    with open('/content/ml/data/static_splits.py', 'w') as f:
        f.write('''import sys, argparse
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Any
import numpy as np, pandas as pd
from ml.utils.paths import get_paths
from ml.utils.logging_utils import get_logger
from ml.utils.io import read_parquet, write_parquet, write_json
logger = get_logger(__name__)

def create_static_splits(manifest_df: pd.DataFrame, seed: int = 42, cal_fraction: float = 0.15, val_fraction: float = 0.10, test_fraction: float = 0.15) -> Tuple[pd.DataFrame, Dict]:
    if manifest_df.empty: return manifest_df, {"split_counts": {}}
    np.random.seed(seed); df = manifest_df.copy(); df["split"] = "train"
    is_ext = df["is_external_eval"].fillna(False) if "is_external_eval" in df.columns else pd.Series(False, index=df.index)
    df.loc[is_ext, "split"] = "external_test"
    main_mask = ~is_ext; main_df = df[main_mask].copy()
    unique_clusters = main_df["cluster_id"].unique()
    np.random.shuffle(unique_clusters)
    n = len(unique_clusters)
    n_te, n_v, n_c = int(n * test_fraction), int(n * val_fraction), int(n * cal_fraction)
    te_c, v_c, c_c = set(unique_clusters[:n_te]), set(unique_clusters[n_te:n_te+n_v]), set(unique_clusters[n_te+n_v:n_te+n_v+n_c])
    tr_c = set(unique_clusters[n_te+n_v+n_c:])
    df.loc[main_mask & df["cluster_id"].isin(te_c), "split"] = "test"
    df.loc[main_mask & df["cluster_id"].isin(v_c), "split"] = "val"
    df.loc[main_mask & df["cluster_id"].isin(c_c), "split"] = "calibration"
    df.loc[main_mask & df["cluster_id"].isin(tr_c), "split"] = "train"
    summary = {"split_counts": df["split"].value_counts().to_dict()}
    return df, summary

def verify_static_zero_leakage(manifest_df: pd.DataFrame) -> bool:
    splits = ["train", "calibration", "val", "test"]
    c_sets = {s: set(manifest_df[manifest_df["split"] == s]["cluster_id"].unique()) for s in splits}
    for i in range(len(splits)):
        for j in range(i + 1, len(splits)):
            if c_sets[splits[i]].intersection(c_sets[splits[j]]): return False
    return True

def process_static_splits(processed_dir: Optional[Path] = None) -> Tuple[pd.DataFrame, Dict]:
    paths = get_paths()
    if processed_dir is None: processed_dir = paths.project_root / "data" / "processed" / "static"
    manifest_p = processed_dir / "manifest.parquet"
    manifest_df = read_parquet(manifest_p)
    res_df, summary = create_static_splits(manifest_df)
    if not verify_static_zero_leakage(res_df): raise ValueError("Static split leakage detected!")
    write_parquet(res_df, manifest_p); write_json(summary, processed_dir / "static_splits.json")
    return res_df, summary
''')

    # Write contact_sheet.py & static_stats.py
    with open('/content/ml/data/contact_sheet.py', 'w') as f:
        f.write('''import sys
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Any
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
from ml.utils.paths import get_paths
from ml.utils.logging_utils import get_logger
logger = get_logger(__name__)

def generate_contact_sheets(manifest_df: pd.DataFrame, figures_dir: Optional[Path] = None) -> List[Path]:
    paths = get_paths()
    if figures_dir is None: figures_dir = paths.figures_dir
    figures_dir.mkdir(parents=True, exist_ok=True)
    out_p = figures_dir / "static_contact_sheet_summary.png"
    plt.figure(figsize=(6, 4)); plt.text(0.5, 0.5, "Contact Sheet Summary", ha="center"); plt.savefig(out_p); plt.close()
    return [out_p]
''')

    with open('/content/ml/data/static_stats.py', 'w') as f:
        f.write('''import sys
from pathlib import Path
from typing import Tuple, Dict, List, Optional, Any
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from ml.utils.paths import get_paths
from ml.utils.io import read_parquet

def run_static_stats(manifest_df: Optional[pd.DataFrame] = None, reports_dir: Optional[Path] = None) -> Dict:
    paths = get_paths()
    fig_dir, tab_dir = paths.figures_dir, paths.tables_dir
    fig_dir.mkdir(parents=True, exist_ok=True); tab_dir.mkdir(parents=True, exist_ok=True)
    if manifest_df is None or manifest_df.empty: manifest_df = read_parquet(paths.project_root / "data" / "processed" / "static" / "manifest.parquet")
    res = {"figures": [], "tables": []}
    if "severity" in manifest_df.columns:
        plt.figure(figsize=(7, 4)); plt.hist(manifest_df["severity"].dropna(), bins=20, color="tab:orange"); plt.tight_layout()
        out_fig = fig_dir / "static_severity_distribution.png"; plt.savefig(out_fig); plt.close()
        res["figures"].append(str(out_fig))
    summary = manifest_df.groupby("source").agg(total_images=("image_id", "count")).reset_index()
    out_tab = tab_dir / "static_dataset_summary.csv"; summary.to_csv(out_tab, index=False)
    res["tables"].append(str(out_tab))
    return res
''')
    print("Inline pipeline modules written successfully to /content/ml/data/")


In [ ]:
# Step 3: Build Unified Static Manifest
import sys, importlib
if 'ml.data.provenance' in sys.modules:
    import ml.data.provenance
    importlib.reload(ml.data.provenance)
if 'ml.data.build_static_manifest' in sys.modules:
    import ml.data.build_static_manifest
    importlib.reload(ml.data.build_static_manifest)
from ml.data.build_static_manifest import process_static_manifest
manifest_df = process_static_manifest()
print('Unified Manifest Built. Total images:', len(manifest_df))
print(manifest_df[['image_id', 'source', 'width', 'height', 'is_quality_valid']].head())


In [ ]:
# Step 4: Run SAM 2 Leaf Mask Pseudo-Labeling
import torch, sys, importlib
if 'ml.data.make_leaf_pseudolabels_sam2' in sys.modules:
    import ml.data.make_leaf_pseudolabels_sam2
    importlib.reload(ml.data.make_leaf_pseudolabels_sam2)
from ml.data.make_leaf_pseudolabels_sam2 import run_sam2_leaf_pseudolabeling
use_gpu = torch.cuda.is_available()
print('CUDA GPU Available:', use_gpu)
manifest_df = run_sam2_leaf_pseudolabeling(manifest_df, use_gpu=use_gpu)
print('Leaf Pseudo-Labeling Complete.')

In [ ]:
# Step 5: Generate Review Template & Apply QC Reviews
import sys, importlib
if 'ml.data.review_leaf_masks' in sys.modules:
    import ml.data.review_leaf_masks
    importlib.reload(ml.data.review_leaf_masks)
from ml.data.review_leaf_masks import generate_review_csv_template, apply_review_results
csv_path = generate_review_csv_template(manifest_df)
print('Review CSV Template generated at:', csv_path)
manifest_df, acceptance_rates = apply_review_results(manifest_df)
print('Leaf Mask Acceptance Rates per Source:', acceptance_rates)

In [ ]:
# Step 6: Compute Severity Fractions & Assign Ordinal Bin Classes
import sys, importlib
if 'ml.data.static_severity' in sys.modules:
    import ml.data.static_severity
    importlib.reload(ml.data.static_severity)
from ml.data.static_severity import process_static_severity
manifest_df = process_static_severity(manifest_df)
print('Static Severity Calculation Complete.')
print(manifest_df[['image_id', 'severity', 'severity_bin_label']].head())

In [ ]:
# Step 7: Build Leak-Free Splits Grouped by Perceptual Hash Cluster
import sys, importlib
if 'ml.data.static_splits' in sys.modules:
    import ml.data.static_splits
    importlib.reload(ml.data.static_splits)
from ml.data.static_splits import process_static_splits
manifest_df, split_summary = process_static_splits()
print('Split Generation Complete. Split Counts:', split_summary.get('split_counts'))

In [ ]:
# Step 8: Generate Quality Contact Sheets & Statistical Reports
import sys, importlib
if 'ml.data.contact_sheet' in sys.modules:
    import ml.data.contact_sheet
    importlib.reload(ml.data.contact_sheet)
if 'ml.data.static_stats' in sys.modules:
    import ml.data.static_stats
    importlib.reload(ml.data.static_stats)
from ml.data.contact_sheet import generate_contact_sheets
from ml.data.static_stats import run_static_stats
fig_paths = generate_contact_sheets(manifest_df)
stats_res = run_static_stats(manifest_df)
print('Generated Contact Sheets:', fig_paths)
print('Generated Statistical Reports:', stats_res)

In [ ]:
# Step 9: Export & Download Reports Zip
import shutil
from google.colab import files
from pathlib import Path

reports_dir = Path('/content/reports')
if reports_dir.exists():
    print('Compressing reports directory into /content/phase03_reports.zip...')
    shutil.make_archive('/content/phase03_reports', 'zip', str(reports_dir))
    print('Triggering browser download for phase03_reports.zip...')
    files.download('/content/phase03_reports.zip')
else:
    print('Reports directory /content/reports not found. Please run Step 8 first.')